## Real Data Estimation

Virtual patient data (Windkessel model) will be used to test with the method. In this set of data, a total of 10 parameters will be used. The input parameters are: [
    "C_p",
    "Za_p",
    "R_p",
    "Emax_rv",
    "Emin_rv",
    "C_s",
    "Za_s",
    "R_s",
    "Emax_lv",
    "Emin_lv"
]

* A previous set of data was used, with only 6 varying parameters. The results can be seen in ../old folder. Some discussions are documented in the latex note.


Import the below functions/libraries vvv

In [1]:
import sys
import os
import pickle

sys.path.insert(0, os.path.abspath(".."))

import numpy as np

from sklearn.preprocessing import StandardScaler

from src.function_library import build_function_library, select_top_power_features, power_features, build_power_library
from src.sparse_interp import sparse_ee_interpretation, save_sparse_result, load_sparse_result, sparse_predict, refine_sparse_result

from npeet import entropy_estimators as ee
import matplotlib.pyplot as plt

#### Pipeline

The function library idea comes from https://doi.org/10.1073/pnas.1517384113. The basic idea is to treat parameter combinations as functions, build a function library, use sparse regression to reduce the dimension of output. A base function library is constructed from the Windkessel parameters using
`build_function_library()`.

The resulting matrix is denoted by:

$$
\Theta_{\mathrm{base}}
$$

Before sparse optimisation, the function library is standardised to avoid scale differences between features.

#### Training

The standardised function library is used to construct a low-dimensional
representation of the Windkessel parameters.

The representation is defined as:

$$
u = \sum_j c_j \widetilde{\Theta}_j(X)
$$

where:

- $\widetilde{\Theta}_j(X)$ is a candidate feature from the standardised
  function library.
- $c_j$ is the coefficient associated with that feature.

The optimisation objective is to maximise the mutual information between
the resulting representation $u$ and the target $y$:

$$
\boxed{
\max_c I(u,y)
}
$$

The optimisation is sparse, so only a subset of the candidate features is
retained in the final representation.

The resulting model is therefore:

$$
\boxed{
u =
\widetilde{\Theta}_{\mathrm{active}}c
}
$$

where `active_indices` identifies the selected features and `coeff`
contains their optimised coefficients.

The result is saved into a `.pkl` file.

### Final data

This dataset contains approximately 800 data points. It was ultimately generated by controlling ventricular pressure based on cleaned_data, after filtering out non-physiological waves.

The experiments involving cleaned_data will be presented in the next section of the notebook.

In [2]:
# loading data
data = np.load(r"D:\Law\25-26\research intern\final_data.npz")
print(data.files)

X_final = data["param"]

param_names = [
    "C_p",
    "Za_p",
    "R_p",
    "Emax_rv",
    "Emin_rv",
    "C_s",
    "Za_s",
    "R_s",
    "Emax_lv",
    "Emin_lv"
]

v_lv_final = data["v_lv"]
v_rv_final = data["v_rv"]

y_v_lv_max_final = np.max(v_lv_final, axis=1)
y_v_lv_min_final = np.min(v_lv_final, axis=1)

y_v_rv_max_final = np.max(v_rv_final, axis=1)
y_v_rv_min_final = np.min(v_rv_final, axis=1)

y_v_lv_mean_final = np.mean(v_lv_final, axis=1)
y_v_rv_mean_final = np.mean(v_rv_final, axis=1)

['p_lv', 'p_rv', 'p_pa', 'v_lv', 'v_rv', 'q_av', 'q_mv', 'q_pv', 'param']


Since this dataset is too small, the training set is also used for power feature selection.

In [3]:
# sampling
rng = np.random.default_rng(42)

n_final_samples = len(X_final)

perm_final = rng.permutation(n_final_samples)

n_final_train = int(0.5 * n_final_samples)
n_final_test   = int(0.5 * n_final_samples)

final_train_idx = perm_final[:n_final_train]
final_test_idx   = perm_final[n_final_train:n_final_train + n_final_test]

# --------------------
# Training set
# --------------------
X_final_train = X_final[final_train_idx]

y_v_lv_max_final_train = y_v_lv_max_final[final_train_idx]
y_v_lv_min_final_train = y_v_lv_min_final[final_train_idx]
y_v_rv_max_final_train = y_v_rv_max_final[final_train_idx]
y_v_rv_min_final_train = y_v_rv_min_final[final_train_idx]
y_v_lv_mean_final_train = y_v_lv_mean_final[final_train_idx]
y_v_rv_mean_final_train = y_v_rv_mean_final[final_train_idx]

# --------------------
# Test set
# --------------------
X_final_test = X_final[final_test_idx]

y_v_lv_max_final_test = y_v_lv_max_final[final_test_idx]
y_v_lv_min_final_test = y_v_lv_min_final[final_test_idx]
y_v_rv_max_final_test = y_v_rv_max_final[final_test_idx]
y_v_rv_min_final_test = y_v_rv_min_final[final_test_idx]
y_v_lv_mean_final_test = y_v_lv_mean_final[final_test_idx]
y_v_rv_mean_final_test = y_v_rv_mean_final[final_test_idx]

# --------------------
# Save split
# --------------------
final_data_split = {
    "train_idx": final_train_idx,
    "test_idx": final_test_idx
}

with open("final_data_split.pkl", "wb") as f:
    pickle.dump(final_data_split, f)

print("Final data split saved.")

Final data split saved.


In [3]:
# load old sampling
with open("final_data_split.pkl", "rb") as f:
    final_data_split = pickle.load(f)

final_train_idx = final_data_split["train_idx"]
final_test_idx = final_data_split["test_idx"]

X_final_train = X_final[final_train_idx]
X_final_test = X_final[final_test_idx]

y_v_lv_max_final_train = y_v_lv_max_final[final_train_idx]
y_v_lv_max_final_test = y_v_lv_max_final[final_test_idx]

y_v_lv_min_final_train = y_v_lv_min_final[final_train_idx]
y_v_lv_min_final_test = y_v_lv_min_final[final_test_idx]

y_v_rv_max_final_train = y_v_rv_max_final[final_train_idx]
y_v_rv_max_final_test = y_v_rv_max_final[final_test_idx]

y_v_rv_min_final_train = y_v_rv_min_final[final_train_idx]
y_v_rv_min_final_test = y_v_rv_min_final[final_test_idx]

y_v_lv_mean_final_train = y_v_lv_mean_final[final_train_idx]
y_v_rv_mean_final_train = y_v_rv_mean_final[final_train_idx]

y_v_lv_mean_final_test = y_v_lv_mean_final[final_test_idx]
y_v_rv_mean_final_test = y_v_rv_mean_final[final_test_idx]

### Cleaned Data

This dataset contains only filtered waveforms, however some of them are still considered as non-physiological. This dataset is used as a secondary verification.

In [23]:
# loading data
data = np.load(r"D:\Law\25-26\research intern\cleaned_data.npz")
print(data.files)

X_cleaned = data["param"]

param_names = [
    "C_p",
    "Za_p",
    "R_p",
    "Emax_rv",
    "Emin_rv",
    "C_s",
    "Za_s",
    "R_s",
    "Emax_lv",
    "Emin_lv"
]

v_lv_cleaned = data["v_lv"]
v_rv_cleaned = data["v_rv"]

y_v_lv_max_cleaned = np.max(v_lv_cleaned, axis=1)
y_v_lv_min_cleaned = np.min(v_lv_cleaned, axis=1)

y_v_rv_max_cleaned = np.max(v_rv_cleaned, axis=1)
y_v_rv_min_cleaned = np.min(v_rv_cleaned, axis=1)

y_v_lv_mean_cleaned = np.mean(v_lv_cleaned, axis=1)
y_v_rv_mean_cleaned = np.mean(v_rv_cleaned, axis=1)

['p_lv', 'p_rv', 'p_pa', 'v_lv', 'v_rv', 'q_av', 'q_mv', 'q_pv', 'param']


In [24]:
# sampling
rng = np.random.default_rng(42)

n_cleaned_samples = len(X_cleaned)

perm_cleaned = rng.permutation(n_cleaned_samples)

n_cleaned_train = int(0.5 * n_cleaned_samples)
n_cleaned_test  = int(0.5 * n_cleaned_samples)

cleaned_train_idx = perm_cleaned[:n_cleaned_train]
cleaned_test_idx  = perm_cleaned[
    n_cleaned_train:n_cleaned_train + n_cleaned_test
]

# --------------------
# Training set
# --------------------
X_cleaned_train = X_cleaned[cleaned_train_idx]

y_v_lv_max_cleaned_train = y_v_lv_max_cleaned[cleaned_train_idx]
y_v_lv_min_cleaned_train = y_v_lv_min_cleaned[cleaned_train_idx]
y_v_rv_max_cleaned_train = y_v_rv_max_cleaned[cleaned_train_idx]
y_v_rv_min_cleaned_train = y_v_rv_min_cleaned[cleaned_train_idx]
y_v_lv_mean_cleaned_train = y_v_lv_mean_cleaned[cleaned_train_idx]
y_v_rv_mean_cleaned_train = y_v_rv_mean_cleaned[cleaned_train_idx]

# --------------------
# Test set
# --------------------
X_cleaned_test = X_cleaned[cleaned_test_idx]

y_v_lv_max_cleaned_test = y_v_lv_max_cleaned[cleaned_test_idx]
y_v_lv_min_cleaned_test = y_v_lv_min_cleaned[cleaned_test_idx]
y_v_rv_max_cleaned_test = y_v_rv_max_cleaned[cleaned_test_idx]
y_v_rv_min_cleaned_test = y_v_rv_min_cleaned[cleaned_test_idx]
y_v_lv_mean_cleaned_test = y_v_lv_mean_cleaned[cleaned_test_idx]
y_v_rv_mean_cleaned_test = y_v_rv_mean_cleaned[cleaned_test_idx]

# --------------------
# Save split
# --------------------
cleaned_data_split = {
    "train_idx": cleaned_train_idx,
    "test_idx": cleaned_test_idx
}

with open("cleaned_data_split.pkl", "wb") as f:
    pickle.dump(cleaned_data_split, f)

print("Cleaned data split saved.")

Cleaned data split saved.


In [25]:
# load old sampling
with open("cleaned_data_split.pkl", "rb") as f:
    cleaned_data_split = pickle.load(f)

cleaned_train_idx = cleaned_data_split["train_idx"]
cleaned_test_idx = cleaned_data_split["test_idx"]

X_cleaned_train = X_cleaned[cleaned_train_idx]
X_cleaned_test = X_cleaned[cleaned_test_idx]

y_v_lv_max_cleaned_train = y_v_lv_max_cleaned[cleaned_train_idx]
y_v_lv_max_cleaned_test = y_v_lv_max_cleaned[cleaned_test_idx]

y_v_lv_min_cleaned_train = y_v_lv_min_cleaned[cleaned_train_idx]
y_v_lv_min_cleaned_test = y_v_lv_min_cleaned[cleaned_test_idx]

y_v_rv_max_cleaned_train = y_v_rv_max_cleaned[cleaned_train_idx]
y_v_rv_max_cleaned_test = y_v_rv_max_cleaned[cleaned_test_idx]

y_v_rv_min_cleaned_train = y_v_rv_min_cleaned[cleaned_train_idx]
y_v_rv_min_cleaned_test = y_v_rv_min_cleaned[cleaned_test_idx]

y_v_lv_mean_cleaned_train = y_v_lv_mean_cleaned[cleaned_train_idx]
y_v_lv_mean_cleaned_test = y_v_lv_mean_cleaned[cleaned_test_idx]

y_v_rv_mean_cleaned_train = y_v_rv_mean_cleaned[cleaned_train_idx]
y_v_rv_mean_cleaned_test = y_v_rv_mean_cleaned[cleaned_test_idx]

### Training

In [5]:
Theta_v_lv_max_final, \
feature_names_v_lv_max_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_lv_max_final = StandardScaler()

Theta_scaled_v_lv_max_final = (
    scaler_v_lv_max_final.fit_transform(
        Theta_v_lv_max_final
    )
)

sparse_result_v_lv_max_final = sparse_ee_interpretation(
    Theta_scaled_v_lv_max_final,
    y_v_lv_max_final_train,
    feature_names_v_lv_max_final,
    scaler_v_lv_max_final,
    threshold=0.05,
    resume=True
)

refined_result_v_lv_max_final = refine_sparse_result(
    sparse_result_v_lv_max_final,
    Theta_scaled_v_lv_max_final,
    y_v_lv_max_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_lv_max_final,
    filename="../results/sparse_result_v_lv_max_final.pkl"
)


===== Iteration 1 =====
MI              : 1.889313
Active features : 75
C_p                                :  2.288869  KEEP
Za_p                               : -2.677541  KEEP
R_p                                :  0.480470  KEEP
Emax_rv                            : -6.095592  KEEP
Emin_rv                            : -0.211767  KEEP
C_s                                :  1.555196  KEEP
Za_s                               :  0.173127  KEEP
R_s                                :  0.266558  KEEP
Emax_lv                            :  8.355386  KEEP
Emin_lv                            :  0.392020  KEEP
C_p^2                              :  0.321021  KEEP
Za_p^2                             :  0.162327  KEEP
R_p^2                              :  2.252863  KEEP
Emax_rv^2                          :  0.161981  KEEP
Emin_rv^2                          :  0.724513  KEEP
C_s^2                              :  1.562166  KEEP
Za_s^2                             :  0.028536  REMOVE
R_s^2                   


===== Refinement Iteration 1 =====
Current MI : 1.952911

C_p                                : coeff=-1.091054  MI(-f)= 1.979152  ΔMI=-0.026241
Za_p                               : coeff= 3.308569  MI(-f)= 1.739725  ΔMI= 0.213186
R_p                                : coeff=-0.621128  MI(-f)= 2.082917  ΔMI=-0.130006
Emax_rv                            : coeff= 6.574513  MI(-f)= 1.310564  ΔMI= 0.642347
Emin_rv                            : coeff= 0.176578  MI(-f)= 2.060687  ΔMI=-0.107776
C_s                                : coeff=-0.408996  MI(-f)= 2.049072  ΔMI=-0.096161
Za_s                               : coeff=-0.374385  MI(-f)= 1.982713  ΔMI=-0.029802
R_s                                : coeff=-0.122014  MI(-f)= 2.067331  ΔMI=-0.114420
Emax_lv                            : coeff=-6.246174  MI(-f)= 1.591139  ΔMI= 0.361772
Emin_lv                            : coeff=-0.110187  MI(-f)= 2.049956  ΔMI=-0.097045
C_p^2                              : coeff=-0.662647  MI(-f)= 2.039628  ΔMI=-0.08

MI after re-optimisation: 2.015240

===== Refinement Iteration 3 =====
Current MI : 2.015240

C_p                                : coeff=-0.849290  MI(-f)= 2.131907  ΔMI=-0.116667
Za_p                               : coeff= 2.893767  MI(-f)= 1.764564  ΔMI= 0.250676
R_p                                : coeff=-1.478827  MI(-f)= 1.999322  ΔMI= 0.015918
Emax_rv                            : coeff= 5.885052  MI(-f)= 1.401968  ΔMI= 0.613272
Emin_rv                            : coeff= 0.127711  MI(-f)= 2.168716  ΔMI=-0.153477
C_s                                : coeff=-0.395258  MI(-f)= 2.166551  ΔMI=-0.151311
Za_s                               : coeff=-0.344112  MI(-f)= 2.088685  ΔMI=-0.073445
R_s                                : coeff=-0.037914  MI(-f)= 2.168117  ΔMI=-0.152877
Emax_lv                            : coeff=-5.722209  MI(-f)= 1.824745  ΔMI= 0.190495
Emin_lv                            : coeff=-0.100103  MI(-f)= 2.168357  ΔMI=-0.153118
C_p^2                              : coeff=-0.

MI after re-optimisation: 2.022847

===== Refinement Iteration 5 =====
Current MI : 2.022847

C_p                                : coeff= 0.892812  MI(-f)= 2.144130  ΔMI=-0.121283
Za_p                               : coeff=-2.829715  MI(-f)= 1.759129  ΔMI= 0.263717
R_p                                : coeff= 1.319609  MI(-f)= 2.104896  ΔMI=-0.082049
Emax_rv                            : coeff=-5.334254  MI(-f)= 1.524541  ΔMI= 0.498305
Emin_rv                            : coeff=-0.165193  MI(-f)= 2.206580  ΔMI=-0.183733
C_s                                : coeff= 0.392414  MI(-f)= 2.219723  ΔMI=-0.196876
Za_s                               : coeff= 0.345193  MI(-f)= 2.133170  ΔMI=-0.110324
R_s                                : coeff=-0.001815  MI(-f)= 2.181599  ΔMI=-0.158752
Emax_lv                            : coeff= 5.719602  MI(-f)= 1.867808  ΔMI= 0.155039
Emin_lv                            : coeff= 0.100476  MI(-f)= 2.193191  ΔMI=-0.170344
C_p^2                              : coeff= 0.

MI after re-optimisation: 2.023769

===== Refinement Iteration 7 =====
Current MI : 2.023769

C_p                                : coeff= 0.792193  MI(-f)= 2.181971  ΔMI=-0.158202
Za_p                               : coeff=-2.872436  MI(-f)= 1.771120  ΔMI= 0.252650
R_p                                : coeff= 1.327887  MI(-f)= 2.100859  ΔMI=-0.077090
Emax_rv                            : coeff=-5.044398  MI(-f)= 1.661531  ΔMI= 0.362239
Emin_rv                            : coeff=-0.196271  MI(-f)= 2.235852  ΔMI=-0.212083
C_s                                : coeff= 0.294091  MI(-f)= 2.242844  ΔMI=-0.219075
Za_s                               : coeff= 0.305216  MI(-f)= 2.193498  ΔMI=-0.169728
R_s                                : coeff= 0.008144  MI(-f)= 2.252568  ΔMI=-0.228799
Emax_lv                            : coeff= 5.746068  MI(-f)= 1.851394  ΔMI= 0.172375
Emin_lv                            : coeff= 0.088411  MI(-f)= 2.249888  ΔMI=-0.226119
C_p^2                              : coeff= 0.

...

MI after re-optimisation: 2.241185

===== Refinement Iteration 42 =====
Current MI : 2.241185

C_p                                : coeff= 1.503785  MI(-f)= 2.195222  ΔMI= 0.045963
R_p                                : coeff= 2.638582  MI(-f)= 2.100568  ΔMI= 0.140617
Emax_rv                            : coeff=-4.020554  MI(-f)= 1.880001  ΔMI= 0.361184
Emin_rv                            : coeff= 1.500733  MI(-f)= 2.384208  ΔMI=-0.143023
C_s                                : coeff= 0.692019  MI(-f)= 2.467517  ΔMI=-0.226332
Za_s                               : coeff= 0.943687  MI(-f)= 2.303556  ΔMI=-0.062371
R_s                                : coeff=-1.218004  MI(-f)= 2.303782  ΔMI=-0.062597
Emax_lv                            : coeff= 4.419176  MI(-f)= 2.367314  ΔMI=-0.126129
C_p^2                              : coeff= 0.866755  MI(-f)= 2.352839  ΔMI=-0.111654
R_p^2                              : coeff= 2.234437  MI(-f)= 2.168137  ΔMI= 0.073048
Emin_rv^2                          : coeff= 2

MI after re-optimisation: 2.284478

===== Refinement Iteration 46 =====
Current MI : 2.284478

C_p                                : coeff= 1.912975  MI(-f)= 2.171444  ΔMI= 0.113035
R_p                                : coeff= 3.124240  MI(-f)= 2.085272  ΔMI= 0.199206
Emax_rv                            : coeff=-3.711458  MI(-f)= 1.903257  ΔMI= 0.381221
Emin_rv                            : coeff= 1.608882  MI(-f)= 2.463950  ΔMI=-0.179472
C_s                                : coeff= 1.840991  MI(-f)= 2.311203  ΔMI=-0.026725
Za_s                               : coeff= 0.828732  MI(-f)= 2.366484  ΔMI=-0.082006
R_s                                : coeff=-1.195910  MI(-f)= 2.401663  ΔMI=-0.117185
Emax_lv                            : coeff= 3.833446  MI(-f)= 2.246391  ΔMI= 0.038087
C_p^2                              : coeff= 0.980994  MI(-f)= 2.407581  ΔMI=-0.123102
R_p^2                              : coeff= 2.285276  MI(-f)= 2.247316  ΔMI= 0.037162
Emin_rv^2                          : coeff= 2

MI after re-optimisation: 2.260263

===== Refinement Iteration 50 =====
Current MI : 2.260263

C_p                                : coeff= 3.064624  MI(-f)= 2.060632  ΔMI= 0.199631
R_p                                : coeff= 2.405195  MI(-f)= 2.251715  ΔMI= 0.008548
Emax_rv                            : coeff=-4.120946  MI(-f)= 1.901841  ΔMI= 0.358423
Emin_rv                            : coeff= 1.611053  MI(-f)= 2.386083  ΔMI=-0.125820
C_s                                : coeff= 1.782861  MI(-f)= 2.339422  ΔMI=-0.079158
Za_s                               : coeff= 0.597207  MI(-f)= 2.433872  ΔMI=-0.173609
R_s                                : coeff=-1.301582  MI(-f)= 2.395608  ΔMI=-0.135344
Emax_lv                            : coeff= 3.867235  MI(-f)= 2.324369  ΔMI=-0.064106
C_p^2                              : coeff= 1.184185  MI(-f)= 2.493326  ΔMI=-0.233063  <-- lowest
R_p^2                              : coeff= 2.344256  MI(-f)= 2.235107  ΔMI= 0.025156
Emin_rv^2                        

MI after re-optimisation: 2.212523

===== Refinement Iteration 55 =====
Current MI : 2.212523

C_p                                : coeff= 3.431198  MI(-f)= 2.004494  ΔMI= 0.208028
R_p                                : coeff= 1.737650  MI(-f)= 2.181014  ΔMI= 0.031509
Emax_rv                            : coeff=-3.932424  MI(-f)= 1.907875  ΔMI= 0.304648
Emin_rv                            : coeff= 1.652273  MI(-f)= 2.275894  ΔMI=-0.063372
C_s                                : coeff= 1.669290  MI(-f)= 2.260825  ΔMI=-0.048302
Emax_lv                            : coeff= 3.941335  MI(-f)= 2.219254  ΔMI=-0.006731
R_p^2                              : coeff= 2.382499  MI(-f)= 2.151651  ΔMI= 0.060872
Emin_rv^2                          : coeff= 1.959014  MI(-f)= 2.263314  ΔMI=-0.050791
C_s^2                              : coeff=-1.490572  MI(-f)= 2.266590  ΔMI=-0.054067
log(C_s)                           : coeff= 3.777208  MI(-f)= 1.979639  ΔMI= 0.232884
log(Emax_lv)                       : coeff= 7

MI after re-optimisation: 2.151780

===== Refinement Iteration 61 =====
Current MI : 2.151780

C_p                                : coeff= 4.537652  MI(-f)= 1.903887  ΔMI= 0.247893
R_p                                : coeff= 2.603820  MI(-f)= 2.193177  ΔMI=-0.041397
Emax_rv                            : coeff=-4.661895  MI(-f)= 1.869091  ΔMI= 0.282689
Emax_lv                            : coeff= 2.897385  MI(-f)= 2.143973  ΔMI= 0.007806
R_p^2                              : coeff= 2.328425  MI(-f)= 2.209033  ΔMI=-0.057254
Emin_rv^2                          : coeff= 2.722442  MI(-f)= 2.022397  ΔMI= 0.129382
log(C_s)                           : coeff= 4.432133  MI(-f)= 1.687713  ΔMI= 0.464067
log(Emax_lv)                       : coeff= 8.232255  MI(-f)= 1.412089  ΔMI= 0.739690
log(Emin_lv)                       : coeff= 2.290625  MI(-f)= 2.209408  ΔMI=-0.057629
C_p*R_p                            : coeff= 1.411830  MI(-f)= 2.350515  ΔMI=-0.198735  <-- lowest
C_s*R_s                          

In [6]:
Theta_v_lv_min_final, \
feature_names_v_lv_min_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_lv_min_final = StandardScaler()

Theta_scaled_v_lv_min_final = (
    scaler_v_lv_min_final.fit_transform(
        Theta_v_lv_min_final
    )
)

sparse_result_v_lv_min_final = sparse_ee_interpretation(
    Theta_scaled_v_lv_min_final,
    y_v_lv_min_final_train,
    feature_names_v_lv_min_final,
    scaler_v_lv_min_final,
    threshold=0.1,
    resume=True
)

refined_result_v_lv_min_final = refine_sparse_result(
    sparse_result_v_lv_min_final,
    Theta_scaled_v_lv_min_final,
    y_v_lv_min_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_lv_min_final,
    filename="../results/sparse_result_v_lv_min_final.pkl"
)


===== Iteration 1 =====
MI              : 0.321116
Active features : 75
C_p                                : -0.217709  KEEP
Za_p                               :  0.214620  KEEP
R_p                                : -0.125382  KEEP
Emax_rv                            :  0.192556  KEEP
Emin_rv                            :  0.077438  REMOVE
C_s                                :  0.200620  KEEP
Za_s                               :  0.155178  KEEP
R_s                                :  0.094677  REMOVE
Emax_lv                            :  0.202522  KEEP
Emin_lv                            :  0.200128  KEEP
C_p^2                              :  0.170380  KEEP
Za_p^2                             :  0.237947  KEEP
R_p^2                              :  0.130677  KEEP
Emax_rv^2                          :  0.199543  KEEP
Emin_rv^2                          :  0.202882  KEEP
C_s^2                              :  0.241299  KEEP
Za_s^2                             :  0.206221  KEEP
R_s^2                 


===== Iteration 4 =====
MI              : 1.255499
Active features : 2
Emax_lv                            :  13.767110  KEEP
Za_p*R_p                           : -0.917854  KEEP

===== Final Sparse Combination =====
Maximum MI : 1.254633
Terms      : 2
Emax_lv                            : 13.767042
Za_p*R_p                           : -0.913109

u ∝
13.7670*Emax_lv + -0.9131*Za_p*R_p

===== Coefficient Pre-screening =====
Original features : 2
Remaining features: 2

Emax_lv                            :  13.767042  KEEP
Za_p*R_p                           : -0.913109  KEEP

===== Refinement Iteration 1 =====
Current MI : 1.254152

Emax_lv                            : coeff= 13.767042  MI(-f)=-0.008307  ΔMI= 1.262459
Za_p*R_p                           : coeff=-0.913109  MI(-f)= 1.498611  ΔMI=-0.244459  <-- lowest

Removing: Za_p*R_p
ΔMI = -0.244459
MI after re-optimisation: 1.160638

===== Refined Sparse Combination =====
Initial terms : 2
Final terms   : 1
Final MI      : 1.160638

Emax

In [4]:
Theta_v_rv_max_final, \
feature_names_v_rv_max_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_rv_max_final = StandardScaler()

Theta_scaled_v_rv_max_final = (
    scaler_v_rv_max_final.fit_transform(
        Theta_v_rv_max_final
    )
)

sparse_result_v_rv_max_final = sparse_ee_interpretation(
    Theta_scaled_v_rv_max_final,
    y_v_rv_max_final_train,
    feature_names_v_rv_max_final,
    scaler_v_rv_max_final,
    threshold=0.1,
    resume=True
)

refined_result_v_rv_max_final = refine_sparse_result(
    sparse_result_v_rv_max_final,
    Theta_scaled_v_rv_max_final,
    y_v_rv_max_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_rv_max_final,
    filename="../results/sparse_result_v_rv_max_final.pkl"
)


===== Iteration 1 =====
MI              : 1.946094
Active features : 75
C_p                                : -4.968072  KEEP
Za_p                               : -0.310720  KEEP
R_p                                : -1.089740  KEEP
Emax_rv                            : -0.675126  KEEP
Emin_rv                            : -0.854407  KEEP
C_s                                : -0.303673  KEEP
Za_s                               : -0.839792  KEEP
R_s                                : -2.394448  KEEP
Emax_lv                            :  1.267595  KEEP
Emin_lv                            : -0.412300  KEEP
C_p^2                              : -0.305568  KEEP
Za_p^2                             : -0.279978  KEEP
R_p^2                              : -0.208709  KEEP
Emax_rv^2                          : -1.006437  KEEP
Emin_rv^2                          : -0.849920  KEEP
C_s^2                              :  1.315698  KEEP
Za_s^2                             : -1.185471  KEEP
R_s^2                     


===== Refinement Iteration 1 =====
Current MI : 1.946289

C_p                                : coeff=-4.968072  MI(-f)= 1.651414  ΔMI= 0.294875
Za_p                               : coeff=-0.310720  MI(-f)= 2.084591  ΔMI=-0.138301
R_p                                : coeff=-1.089740  MI(-f)= 2.067393  ΔMI=-0.121104
Emax_rv                            : coeff=-0.675126  MI(-f)= 2.051100  ΔMI=-0.104811
Emin_rv                            : coeff=-0.854407  MI(-f)= 2.027011  ΔMI=-0.080722
C_s                                : coeff=-0.303673  MI(-f)= 2.001604  ΔMI=-0.055314
Za_s                               : coeff=-0.839792  MI(-f)= 2.065766  ΔMI=-0.119477
R_s                                : coeff=-2.394448  MI(-f)= 2.024849  ΔMI=-0.078559
Emax_lv                            : coeff= 1.267595  MI(-f)= 2.006190  ΔMI=-0.059900
Emin_lv                            : coeff=-0.412300  MI(-f)= 2.077122  ΔMI=-0.130832
C_p^2                              : coeff=-0.305568  MI(-f)= 2.057744  ΔMI=-0.11

MI after re-optimisation: 1.949947

===== Refinement Iteration 3 =====
Current MI : 1.949947

C_p                                : coeff=-5.173297  MI(-f)= 1.687663  ΔMI= 0.262285
Za_p                               : coeff=-0.313820  MI(-f)= 2.197215  ΔMI=-0.247268
R_p                                : coeff=-1.178262  MI(-f)= 2.170264  ΔMI=-0.220317
Emax_rv                            : coeff=-0.276224  MI(-f)= 2.220733  ΔMI=-0.270786
Emin_rv                            : coeff=-0.855284  MI(-f)= 2.098579  ΔMI=-0.148631
C_s                                : coeff=-0.319546  MI(-f)= 2.162442  ΔMI=-0.212495
Za_s                               : coeff=-0.852031  MI(-f)= 2.121243  ΔMI=-0.171295
R_s                                : coeff=-2.388768  MI(-f)= 2.114144  ΔMI=-0.164197
Emax_lv                            : coeff= 1.145509  MI(-f)= 2.121626  ΔMI=-0.171678
Emin_lv                            : coeff=-0.425715  MI(-f)= 2.180696  ΔMI=-0.230748
C_p^2                              : coeff=-0.

MI after re-optimisation: 1.971555

===== Refinement Iteration 5 =====
Current MI : 1.971555

C_p                                : coeff=-5.213073  MI(-f)= 1.642979  ΔMI= 0.328577
Za_p                               : coeff=-0.408407  MI(-f)= 2.253759  ΔMI=-0.282204
R_p                                : coeff=-1.088073  MI(-f)= 2.172465  ΔMI=-0.200909
Emax_rv                            : coeff= 0.074404  MI(-f)= 2.229563  ΔMI=-0.258008
Emin_rv                            : coeff=-1.218156  MI(-f)= 2.131874  ΔMI=-0.160318
C_s                                : coeff= 0.219280  MI(-f)= 2.234957  ΔMI=-0.263402
Za_s                               : coeff=-0.910372  MI(-f)= 2.153932  ΔMI=-0.182377
R_s                                : coeff=-2.407385  MI(-f)= 2.115793  ΔMI=-0.144237
Emax_lv                            : coeff= 1.533149  MI(-f)= 2.120822  ΔMI=-0.149267
Emin_lv                            : coeff= 0.014856  MI(-f)= 2.241963  ΔMI=-0.270408
C_p^2                              : coeff=-0.

MI after re-optimisation: 1.978097

===== Refinement Iteration 7 =====
Current MI : 1.978097

C_p                                : coeff=-5.163944  MI(-f)= 1.596126  ΔMI= 0.381971
Za_p                               : coeff=-0.697272  MI(-f)= 2.173602  ΔMI=-0.195505
R_p                                : coeff=-1.638951  MI(-f)= 2.161703  ΔMI=-0.183606
Emax_rv                            : coeff= 0.017711  MI(-f)= 2.164399  ΔMI=-0.186302
Emin_rv                            : coeff=-0.830692  MI(-f)= 2.093070  ΔMI=-0.114972
C_s                                : coeff= 0.224438  MI(-f)= 2.162109  ΔMI=-0.184012
Za_s                               : coeff=-0.890608  MI(-f)= 2.152938  ΔMI=-0.174841
R_s                                : coeff=-2.396738  MI(-f)= 2.094579  ΔMI=-0.116481
Emax_lv                            : coeff= 1.521774  MI(-f)= 2.104258  ΔMI=-0.126160
Emin_lv                            : coeff= 0.017459  MI(-f)= 2.170668  ΔMI=-0.192571
C_p^2                              : coeff=-0.

...

MI after re-optimisation: 2.398688

===== Refinement Iteration 46 =====
Current MI : 2.398688

C_p                                : coeff= 5.816485  MI(-f)= 1.969677  ΔMI= 0.429012
Za_p                               : coeff= 2.515573  MI(-f)= 2.527541  ΔMI=-0.128852
Emin_rv                            : coeff= 5.287619  MI(-f)= 1.987411  ΔMI= 0.411277
C_s                                : coeff= 1.734920  MI(-f)= 2.697120  ΔMI=-0.298431
Za_s                               : coeff= 5.265129  MI(-f)= 2.114822  ΔMI= 0.283866
R_s                                : coeff= 2.513242  MI(-f)= 2.561666  ΔMI=-0.162977
Emin_lv                            : coeff= 2.373219  MI(-f)= 2.552981  ΔMI=-0.154293
Emin_rv^2                          : coeff= 1.762807  MI(-f)= 2.668302  ΔMI=-0.269613
R_s^2                              : coeff=-2.837108  MI(-f)= 2.537018  ΔMI=-0.138329
Emax_lv^2                          : coeff= 3.008130  MI(-f)= 2.596944  ΔMI=-0.198256
log(C_p)                           : coeff= 2

MI after re-optimisation: 2.405765

===== Refinement Iteration 50 =====
Current MI : 2.405765

C_p                                : coeff= 7.390117  MI(-f)= 1.714002  ΔMI= 0.691763
Za_p                               : coeff= 2.785356  MI(-f)= 2.473888  ΔMI=-0.068124
Emin_rv                            : coeff= 4.757285  MI(-f)= 2.146299  ΔMI= 0.259466
C_s                                : coeff= 2.481936  MI(-f)= 2.726745  ΔMI=-0.320980
Za_s                               : coeff= 5.623708  MI(-f)= 2.051622  ΔMI= 0.354143
R_s                                : coeff= 1.327829  MI(-f)= 2.792286  ΔMI=-0.386521
Emin_lv                            : coeff= 1.872033  MI(-f)= 2.669701  ΔMI=-0.263936
Emin_rv^2                          : coeff= 1.782517  MI(-f)= 2.679677  ΔMI=-0.273912
R_s^2                              : coeff=-2.855882  MI(-f)= 2.647838  ΔMI=-0.242073
Emax_lv^2                          : coeff= 2.886645  MI(-f)= 2.527465  ΔMI=-0.121701
log(C_p)                           : coeff= 2

MI after re-optimisation: 2.458687

===== Refinement Iteration 54 =====
Current MI : 2.458687

C_p                                : coeff= 6.228533  MI(-f)= 1.951484  ΔMI= 0.507202
Za_p                               : coeff= 2.291665  MI(-f)= 2.560589  ΔMI=-0.101902
Emin_rv                            : coeff= 4.118686  MI(-f)= 2.132123  ΔMI= 0.326563
C_s                                : coeff= 3.654504  MI(-f)= 2.578459  ΔMI=-0.119773
Za_s                               : coeff= 5.388674  MI(-f)= 2.085005  ΔMI= 0.373681
Emin_lv                            : coeff= 2.034265  MI(-f)= 2.752244  ΔMI=-0.293557
Emin_rv^2                          : coeff= 2.101424  MI(-f)= 2.599434  ΔMI=-0.140748
R_s^2                              : coeff=-2.374272  MI(-f)= 2.717053  ΔMI=-0.258367
Emax_lv^2                          : coeff= 3.330606  MI(-f)= 2.511070  ΔMI=-0.052384
log(C_p)                           : coeff= 2.415421  MI(-f)= 2.664238  ΔMI=-0.205551
log(Emax_rv)                       : coeff= 5

MI after re-optimisation: 2.359489

===== Refinement Iteration 59 =====
Current MI : 2.359489

C_p                                : coeff= 4.304242  MI(-f)= 2.156412  ΔMI= 0.203078
Emin_rv                            : coeff= 4.440566  MI(-f)= 2.162194  ΔMI= 0.197295
C_s                                : coeff= 2.998418  MI(-f)= 2.500885  ΔMI=-0.141396
Za_s                               : coeff= 5.796860  MI(-f)= 1.903059  ΔMI= 0.456430
Emin_rv^2                          : coeff= 2.014943  MI(-f)= 2.536903  ΔMI=-0.177414
Emax_lv^2                          : coeff= 2.026631  MI(-f)= 2.524787  ΔMI=-0.165298
log(C_p)                           : coeff= 3.185188  MI(-f)= 2.333590  ΔMI= 0.025899
log(Emax_rv)                       : coeff= 7.055623  MI(-f)= 1.714127  ΔMI= 0.645363
log(R_s)                           : coeff= 3.956162  MI(-f)= 2.309341  ΔMI= 0.050148
log(Emax_lv)                       : coeff=-4.010405  MI(-f)= 2.180947  ΔMI= 0.178542
C_p*R_p                            : coeff= 3

MI after re-optimisation: 2.211221

===== Refinement Iteration 65 =====
Current MI : 2.211221

Emin_rv                            : coeff= 6.337005  MI(-f)= 1.856470  ΔMI= 0.354751
Za_s                               : coeff= 6.788976  MI(-f)= 1.819233  ΔMI= 0.391988
log(C_p)                           : coeff= 5.395828  MI(-f)= 1.840198  ΔMI= 0.371023
log(Emax_rv)                       : coeff= 7.919040  MI(-f)= 1.650174  ΔMI= 0.561047
log(R_s)                           : coeff= 3.138088  MI(-f)= 2.439696  ΔMI=-0.228475  <-- lowest
log(Emax_lv)                       : coeff=-4.264651  MI(-f)= 2.079647  ΔMI= 0.131574
C_p*R_p                            : coeff= 4.086736  MI(-f)= 2.224712  ΔMI=-0.013491
C_p*Emax_rv                        : coeff= 5.929046  MI(-f)= 1.928909  ΔMI= 0.282312
R_p*Za_s                           : coeff=-4.834437  MI(-f)= 1.976969  ΔMI= 0.234252
Emax_rv*R_s                        : coeff= 5.977335  MI(-f)= 2.242075  ΔMI=-0.030854
C_s*R_s                          

In [7]:
Theta_v_rv_min_final, \
feature_names_v_rv_min_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_rv_min_final = StandardScaler()

Theta_scaled_v_rv_min_final = (
    scaler_v_rv_min_final.fit_transform(
        Theta_v_rv_min_final
    )
)

sparse_result_v_rv_min_final = sparse_ee_interpretation(
    Theta_scaled_v_rv_min_final,
    y_v_rv_min_final_train,
    feature_names_v_rv_min_final,
    scaler_v_rv_min_final,
    threshold=0.1,
    resume=True
)

refined_result_v_rv_min_final = refine_sparse_result(
    sparse_result_v_rv_min_final,
    Theta_scaled_v_rv_min_final,
    y_v_rv_min_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_rv_min_final,
    filename="../results/sparse_result_v_rv_min_final.pkl"
)


===== Iteration 1 =====
MI              : 0.541674
Active features : 75
C_p                                :  1.897586  KEEP
Za_p                               :  0.374987  KEEP
R_p                                :  0.893710  KEEP
Emax_rv                            :  0.412629  KEEP
Emin_rv                            :  0.373493  KEEP
C_s                                :  0.372841  KEEP
Za_s                               :  0.372522  KEEP
R_s                                :  0.339611  KEEP
Emax_lv                            :  0.128440  KEEP
Emin_lv                            :  0.121852  KEEP
C_p^2                              :  0.549093  KEEP
Za_p^2                             :  0.380904  KEEP
R_p^2                              :  0.321761  KEEP
Emax_rv^2                          :  0.373517  KEEP
Emin_rv^2                          :  0.373862  KEEP
C_s^2                              :  1.316297  KEEP
Za_s^2                             :  0.316975  KEEP
R_s^2                     


===== Iteration 4 =====
MI              : 1.403864
Active features : 16
C_p                                :  4.079107  KEEP
Emax_rv                            :  13.930457  KEEP
Za_s                               : -1.660523  KEEP
R_s                                :  0.892493  KEEP
Emax_rv^2                          :  3.079415  KEEP
Emin_rv^2                          :  0.506411  KEEP
C_s^2                              :  0.868234  KEEP
log(R_s)                           :  0.727286  KEEP
C_p*C_s                            :  1.022559  KEEP
C_p*Za_s                           :  0.759687  KEEP
Za_p*Emax_rv                       :  1.020155  KEEP
Za_p*Emin_rv                       :  0.862866  KEEP
Za_p*C_s                           :  0.669013  KEEP
C_s*Emin_lv                        :  0.757784  KEEP
Za_s*R_s                           :  0.680942  KEEP
Za_s*Emax_lv                       : -0.008514  REMOVE

===== Iteration 5 =====
MI              : 1.418602
Active features : 15
C_p

MI after re-optimisation: 1.507684

===== Refinement Iteration 5 =====
Current MI : 1.507684

C_p                                : coeff= 6.195580  MI(-f)= 1.204329  ΔMI= 0.303355
Emax_rv                            : coeff= 12.096698  MI(-f)= 0.716256  ΔMI= 0.791428
Za_s                               : coeff=-4.872984  MI(-f)= 1.150116  ΔMI= 0.357568
Emax_rv^2                          : coeff= 5.044227  MI(-f)= 1.428620  ΔMI= 0.079064
Emin_rv^2                          : coeff= 1.732881  MI(-f)= 1.385749  ΔMI= 0.121935
C_s^2                              : coeff= 0.079711  MI(-f)= 1.594463  ΔMI=-0.086779  <-- lowest
C_p*C_s                            : coeff= 3.895921  MI(-f)= 1.233704  ΔMI= 0.273980
Za_p*Emin_rv                       : coeff= 1.623503  MI(-f)= 1.522097  ΔMI=-0.014413
Za_s*R_s                           : coeff= 5.342143  MI(-f)= 1.386327  ΔMI= 0.121356

Removing: C_s^2
ΔMI = -0.086779
MI after re-optimisation: 1.506138

===== Refinement Iteration 6 =====
Current MI : 1.

In [8]:
Theta_v_lv_mean_final, \
feature_names_v_lv_mean_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_lv_mean_final = StandardScaler()

Theta_scaled_v_lv_mean_final = (
    scaler_v_lv_mean_final.fit_transform(
        Theta_v_lv_mean_final
    )
)

sparse_result_v_lv_mean_final = sparse_ee_interpretation(
    Theta_scaled_v_lv_mean_final,
    y_v_lv_mean_final_train,
    feature_names_v_lv_mean_final,
    scaler_v_lv_mean_final,
    threshold=0.1,
    resume=True
)

refined_result_v_lv_mean_final = refine_sparse_result(
    sparse_result_v_lv_mean_final,
    Theta_scaled_v_lv_mean_final,
    y_v_lv_mean_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_lv_mean_final,
    filename="../results/sparse_result_v_lv_mean_final.pkl"
)


===== Iteration 1 =====
MI              : 1.862178
Active features : 75
C_p                                :  0.890297  KEEP
Za_p                               :  0.150256  KEEP
R_p                                :  0.108700  KEEP
Emax_rv                            :  0.104258  KEEP
Emin_rv                            :  0.110801  KEEP
C_s                                :  0.107915  KEEP
Za_s                               :  0.110610  KEEP
R_s                                :  0.091208  REMOVE
Emax_lv                            :  0.140113  KEEP
Emin_lv                            :  0.112926  KEEP
C_p^2                              :  0.109423  KEEP
Za_p^2                             :  0.110764  KEEP
R_p^2                              :  0.113421  KEEP
Emax_rv^2                          :  0.044401  REMOVE
Emin_rv^2                          :  0.264660  KEEP
C_s^2                              :  0.116872  KEEP
Za_s^2                             :  0.019413  REMOVE
R_s^2               


===== Iteration 4 =====
MI              : 1.950451
Active features : 56
C_p                                :  0.614783  KEEP
Za_p                               : -1.118220  KEEP
R_p                                :  0.170463  KEEP
Emax_rv                            : -1.758268  KEEP
Emin_rv                            :  0.228709  KEEP
C_s                                :  0.197761  KEEP
Za_s                               :  0.194058  KEEP
Emax_lv                            :  9.837993  KEEP
C_p^2                              : -0.198052  KEEP
Za_p^2                             :  0.258589  KEEP
R_s^2                              : -0.292164  KEEP
Emin_lv^2                          :  0.283914  KEEP
log(C_p)                           :  0.178265  KEEP
log(R_p)                           :  0.175188  KEEP
log(C_s)                           :  0.775563  KEEP
log(Emax_lv)                       :  1.860141  KEEP
log(Emin_lv)                       :  0.172747  KEEP
C_p*Za_p                  


===== Iteration 7 =====
MI              : 1.889857
Active features : 49
C_p                                :  0.220756  KEEP
Za_p                               :  0.162513  KEEP
Emax_rv                            : -0.634131  KEEP
Emin_rv                            :  0.167482  KEEP
C_s                                :  0.167909  KEEP
Emax_lv                            :  10.589788  KEEP
C_p^2                              :  0.167421  KEEP
Za_p^2                             : -0.873992  KEEP
Emin_lv^2                          :  0.172988  KEEP
log(C_p)                           :  0.130151  KEEP
log(R_p)                           :  0.613674  KEEP
log(C_s)                           :  0.390788  KEEP
log(Emax_lv)                       :  0.804356  KEEP
log(Emin_lv)                       :  0.239392  KEEP
C_p*Za_p                           :  0.173494  KEEP
C_p*R_p                            :  1.390723  KEEP
C_p*Emax_rv                        :  0.168376  KEEP
C_p*Emin_rv              


===== Refinement Iteration 1 =====
Current MI : 1.889857

C_p                                : coeff= 0.220756  MI(-f)= 2.022179  ΔMI=-0.132321
Za_p                               : coeff= 0.162513  MI(-f)= 2.045918  ΔMI=-0.156061
Emax_rv                            : coeff=-0.634131  MI(-f)= 1.900234  ΔMI=-0.010376
Emin_rv                            : coeff= 0.167482  MI(-f)= 2.010048  ΔMI=-0.120191
C_s                                : coeff= 0.167909  MI(-f)= 2.015029  ΔMI=-0.125172
Emax_lv                            : coeff= 10.589788  MI(-f)= 0.842139  ΔMI= 1.047719
C_p^2                              : coeff= 0.167421  MI(-f)= 2.029391  ΔMI=-0.139534
Za_p^2                             : coeff=-0.873992  MI(-f)= 1.936529  ΔMI=-0.046672
Emin_lv^2                          : coeff= 0.172988  MI(-f)= 1.979791  ΔMI=-0.089934
log(C_p)                           : coeff= 0.130151  MI(-f)= 2.025360  ΔMI=-0.135503
log(R_p)                           : coeff= 0.613674  MI(-f)= 2.043651  ΔMI=-0.1

MI after re-optimisation: 1.909595

===== Refinement Iteration 3 =====
Current MI : 1.909595

Za_p                               : coeff=-0.000692  MI(-f)= 2.020159  ΔMI=-0.110564
Emax_rv                            : coeff=-0.929573  MI(-f)= 1.919931  ΔMI=-0.010337
Emin_rv                            : coeff= 0.143359  MI(-f)= 2.005937  ΔMI=-0.096343
C_s                                : coeff= 0.170479  MI(-f)= 1.986733  ΔMI=-0.077139
Emax_lv                            : coeff= 9.509244  MI(-f)= 1.214602  ΔMI= 0.694992
C_p^2                              : coeff= 1.351021  MI(-f)= 2.041596  ΔMI=-0.132001
Za_p^2                             : coeff=-0.809758  MI(-f)= 2.015870  ΔMI=-0.106276
Emin_lv^2                          : coeff= 0.204153  MI(-f)= 1.975498  ΔMI=-0.065903
log(C_p)                           : coeff= 0.155390  MI(-f)= 2.010999  ΔMI=-0.101405
log(R_p)                           : coeff= 1.029474  MI(-f)= 2.119791  ΔMI=-0.210196  <-- lowest
log(C_s)                          

...

MI after re-optimisation: 2.244239

===== Refinement Iteration 20 =====
Current MI : 2.244239

Emax_rv                            : coeff=-1.834441  MI(-f)= 2.152997  ΔMI= 0.091241
Emin_rv                            : coeff= 0.445614  MI(-f)= 2.262955  ΔMI=-0.018716
Emax_lv                            : coeff= 3.715273  MI(-f)= 2.282087  ΔMI=-0.037848
C_p^2                              : coeff= 0.880426  MI(-f)= 2.309086  ΔMI=-0.064847
Za_p^2                             : coeff=-1.088730  MI(-f)= 2.375451  ΔMI=-0.131213  <-- lowest
Emin_lv^2                          : coeff= 0.147362  MI(-f)= 2.325628  ΔMI=-0.081389
log(C_p)                           : coeff= 0.567145  MI(-f)= 2.318369  ΔMI=-0.074130
log(Emax_lv)                       : coeff= 4.798968  MI(-f)= 2.133440  ΔMI= 0.110798
log(Emin_lv)                       : coeff= 0.536585  MI(-f)= 2.303593  ΔMI=-0.059355
C_p*Za_p                           : coeff= 0.490489  MI(-f)= 2.239443  ΔMI= 0.004795
C_p*Emax_rv                      

MI after re-optimisation: 2.267283

===== Refinement Iteration 24 =====
Current MI : 2.267283

Emax_rv                            : coeff=-1.906656  MI(-f)= 2.162784  ΔMI= 0.104499
Emin_rv                            : coeff= 0.471450  MI(-f)= 2.408236  ΔMI=-0.140953
Emax_lv                            : coeff= 3.510068  MI(-f)= 2.336154  ΔMI=-0.068871
C_p^2                              : coeff= 1.146652  MI(-f)= 2.441777  ΔMI=-0.174494  <-- lowest
Emin_lv^2                          : coeff= 0.181235  MI(-f)= 2.351320  ΔMI=-0.084037
log(C_p)                           : coeff= 0.346421  MI(-f)= 2.427028  ΔMI=-0.159745
log(Emax_lv)                       : coeff= 4.847155  MI(-f)= 2.131585  ΔMI= 0.135698
log(Emin_lv)                       : coeff= 0.580360  MI(-f)= 2.332163  ΔMI=-0.064880
C_p*Emax_rv                        : coeff= 1.259207  MI(-f)= 2.416981  ΔMI=-0.149698
C_p*C_s                            : coeff= 0.349248  MI(-f)= 2.433318  ΔMI=-0.166035
C_p*Za_s                         

MI after re-optimisation: 2.372431

===== Refinement Iteration 28 =====
Current MI : 2.372431

Emax_rv                            : coeff=-2.417315  MI(-f)= 1.971048  ΔMI= 0.401383
Emax_lv                            : coeff= 3.667375  MI(-f)= 2.401021  ΔMI=-0.028590
Emin_lv^2                          : coeff= 0.081843  MI(-f)= 2.511201  ΔMI=-0.138770
log(Emax_lv)                       : coeff= 5.048264  MI(-f)= 2.296990  ΔMI= 0.075441
log(Emin_lv)                       : coeff= 0.590585  MI(-f)= 2.423158  ΔMI=-0.050727
C_p*Emax_rv                        : coeff= 2.181518  MI(-f)= 2.109871  ΔMI= 0.262560
C_p*C_s                            : coeff= 0.545701  MI(-f)= 2.500529  ΔMI=-0.128098
C_p*Za_s                           : coeff= 0.313704  MI(-f)= 2.516110  ΔMI=-0.143679  <-- lowest
Za_p*Emin_rv                       : coeff=-0.388790  MI(-f)= 2.465279  ΔMI=-0.092848
Za_p*C_s                           : coeff=-0.601868  MI(-f)= 2.479860  ΔMI=-0.107429
Za_p*Emin_lv                     

MI after re-optimisation: 2.420862

===== Refinement Iteration 33 =====
Current MI : 2.420862

Emax_rv                            : coeff=-2.629025  MI(-f)= 2.082920  ΔMI= 0.337942
Emax_lv                            : coeff= 0.852955  MI(-f)= 2.554949  ΔMI=-0.134087
log(Emax_lv)                       : coeff= 6.829949  MI(-f)= 1.906247  ΔMI= 0.514615
C_p*Emax_rv                        : coeff= 2.492715  MI(-f)= 2.188999  ΔMI= 0.231863
C_p*C_s                            : coeff= 1.009743  MI(-f)= 2.524688  ΔMI=-0.103826
Za_p*Emin_rv                       : coeff=-0.713028  MI(-f)= 2.565512  ΔMI=-0.144650
R_p*C_s                            : coeff=-0.371306  MI(-f)= 2.591806  ΔMI=-0.170944  <-- lowest
R_p*Emax_lv                        : coeff= 5.230178  MI(-f)= 1.976345  ΔMI= 0.444517
R_p*Emin_lv                        : coeff= 0.601005  MI(-f)= 2.482168  ΔMI=-0.061306
Emin_rv*C_s                        : coeff= 0.546330  MI(-f)= 2.529622  ΔMI=-0.108760
Emin_rv*Za_s                     

MI after re-optimisation: 2.379640

===== Refinement Iteration 39 =====
Current MI : 2.379640

Emax_rv                            : coeff=-3.376327  MI(-f)= 1.940791  ΔMI= 0.438850
log(Emax_lv)                       : coeff= 7.637726  MI(-f)= 1.718842  ΔMI= 0.660799
C_p*Emax_rv                        : coeff= 3.403322  MI(-f)= 1.963525  ΔMI= 0.416116
R_p*Emax_lv                        : coeff= 5.262959  MI(-f)= 2.012905  ΔMI= 0.366736
R_p*Emin_lv                        : coeff= 0.655117  MI(-f)= 2.561749  ΔMI=-0.182108  <-- lowest
Emin_rv*C_s                        : coeff= 0.799376  MI(-f)= 2.507196  ΔMI=-0.127556
Emin_rv*Za_s                       : coeff= 1.935277  MI(-f)= 2.446873  ΔMI=-0.067233
C_s*R_s                            : coeff= 4.166572  MI(-f)= 1.890724  ΔMI= 0.488916
Za_s*R_s                           : coeff=-2.010722  MI(-f)= 2.308048  ΔMI= 0.071593
R_s*Emax_lv                        : coeff=-2.745739  MI(-f)= 2.229186  ΔMI= 0.150455
Emax_lv*Emin_lv                  

In [9]:
Theta_v_rv_mean_final, \
feature_names_v_rv_mean_final = build_function_library(
    X_final_train,
    param_names
)

scaler_v_rv_mean_final = StandardScaler()

Theta_scaled_v_rv_mean_final = (
    scaler_v_rv_mean_final.fit_transform(
        Theta_v_rv_mean_final
    )
)

sparse_result_v_rv_mean_final = sparse_ee_interpretation(
    Theta_scaled_v_rv_mean_final,
    y_v_rv_mean_final_train,
    feature_names_v_rv_mean_final,
    scaler_v_rv_mean_final,
    threshold=0.1,
    resume=True
)

refined_result_v_rv_mean_final = refine_sparse_result(
    sparse_result_v_rv_mean_final,
    Theta_scaled_v_rv_mean_final,
    y_v_rv_mean_final_train,
    coefficient_threshold=0.1,
    importance_threshold=0,
)

save_sparse_result(
    refined_result_v_rv_mean_final,
    filename="../results/sparse_result_v_rv_mean_final.pkl"
)


===== Iteration 1 =====
MI              : 1.249304
Active features : 75
C_p                                :  0.061285  REMOVE
Za_p                               :  0.099460  REMOVE
R_p                                :  0.099359  REMOVE
Emax_rv                            :  13.221489  KEEP
Emin_rv                            :  0.115645  KEEP
C_s                                : -0.074965  REMOVE
Za_s                               : -1.008952  KEEP
R_s                                :  0.099170  REMOVE
Emax_lv                            :  0.201562  KEEP
Emin_lv                            :  0.100199  KEEP
C_p^2                              :  2.942578  KEEP
Za_p^2                             : -0.350753  KEEP
R_p^2                              : -1.251696  KEEP
Emax_rv^2                          :  2.983409  KEEP
Emin_rv^2                          :  0.395334  KEEP
C_s^2                              :  0.338660  KEEP
Za_s^2                             :  0.101704  KEEP
R_s^2          


===== Refinement Iteration 1 =====
Current MI : 1.813363

Emax_rv                            : coeff= 16.102293  MI(-f)= 0.789828  ΔMI= 1.023535
Emin_rv                            : coeff= 0.476207  MI(-f)= 1.974070  ΔMI=-0.160707
Emax_lv                            : coeff=-0.530208  MI(-f)= 1.971570  ΔMI=-0.158207
Emin_lv                            : coeff=-0.645452  MI(-f)= 1.957262  ΔMI=-0.143899
C_p^2                              : coeff= 4.546119  MI(-f)= 1.572106  ΔMI= 0.241257
Za_p^2                             : coeff=-0.752075  MI(-f)= 1.902183  ΔMI=-0.088819
R_p^2                              : coeff=-4.316349  MI(-f)= 1.649274  ΔMI= 0.164090
Emax_rv^2                          : coeff=-8.008621  MI(-f)= 1.707573  ΔMI= 0.105790
Emin_rv^2                          : coeff= 0.715176  MI(-f)= 1.957689  ΔMI=-0.144326
C_s^2                              : coeff= 0.314547  MI(-f)= 2.024776  ΔMI=-0.211413
Za_s^2                             : coeff= 1.465201  MI(-f)= 1.883726  ΔMI=-0.0

MI after re-optimisation: 1.904726

===== Refinement Iteration 5 =====
Current MI : 1.904726

Emax_rv                            : coeff= 13.856948  MI(-f)= 0.917693  ΔMI= 0.987034
Emin_rv                            : coeff= 2.001789  MI(-f)= 1.945015  ΔMI=-0.040289
Emax_lv                            : coeff=-1.031461  MI(-f)= 1.965121  ΔMI=-0.060395
C_p^2                              : coeff= 4.628136  MI(-f)= 1.744734  ΔMI= 0.159992
Za_p^2                             : coeff=-0.630030  MI(-f)= 2.156986  ΔMI=-0.252260  <-- lowest
R_p^2                              : coeff=-4.677655  MI(-f)= 1.513179  ΔMI= 0.391547
Emax_rv^2                          : coeff=-7.207088  MI(-f)= 1.886554  ΔMI= 0.018172
C_s^2                              : coeff=-0.253578  MI(-f)= 2.123649  ΔMI=-0.218923
Za_s^2                             : coeff= 1.509857  MI(-f)= 2.003856  ΔMI=-0.099130
R_s^2                              : coeff= 4.170949  MI(-f)= 1.996434  ΔMI=-0.091708
log(C_p)                         

MI after re-optimisation: 1.893807

===== Refinement Iteration 9 =====
Current MI : 1.893807

Emax_rv                            : coeff= 13.147553  MI(-f)= 1.038968  ΔMI= 0.854839
Emin_rv                            : coeff= 2.010306  MI(-f)= 1.931427  ΔMI=-0.037620
Emax_lv                            : coeff=-1.039512  MI(-f)= 1.958987  ΔMI=-0.065180
C_p^2                              : coeff= 4.228814  MI(-f)= 1.648639  ΔMI= 0.245168
R_p^2                              : coeff=-4.550511  MI(-f)= 1.613671  ΔMI= 0.280135
Emax_rv^2                          : coeff=-5.937812  MI(-f)= 1.906437  ΔMI=-0.012630
Za_s^2                             : coeff= 0.998072  MI(-f)= 1.955601  ΔMI=-0.061794
R_s^2                              : coeff= 4.504531  MI(-f)= 1.777545  ΔMI= 0.116262
log(C_p)                           : coeff= 2.933121  MI(-f)= 1.842392  ΔMI= 0.051415
log(Emax_rv)                       : coeff= 6.895884  MI(-f)= 1.919948  ΔMI=-0.026141
log(C_s)                           : coeff= 3

MI after re-optimisation: 1.947907

===== Refinement Iteration 14 =====
Current MI : 1.947907

Emax_rv                            : coeff= 9.653258  MI(-f)= 1.300763  ΔMI= 0.647144
Emin_rv                            : coeff= 1.859121  MI(-f)= 2.020595  ΔMI=-0.072688
Emax_lv                            : coeff=-2.115041  MI(-f)= 2.117492  ΔMI=-0.169585
C_p^2                              : coeff= 4.495368  MI(-f)= 1.832085  ΔMI= 0.115822
R_p^2                              : coeff=-5.656474  MI(-f)= 1.640570  ΔMI= 0.307337
Emax_rv^2                          : coeff=-4.383642  MI(-f)= 2.169185  ΔMI=-0.221278
R_s^2                              : coeff= 4.625272  MI(-f)= 2.055851  ΔMI=-0.107944
log(C_p)                           : coeff= 4.417265  MI(-f)= 1.873979  ΔMI= 0.073928
log(Emax_rv)                       : coeff= 7.625523  MI(-f)= 1.552058  ΔMI= 0.395849
log(C_s)                           : coeff= 3.764454  MI(-f)= 1.898358  ΔMI= 0.049549
C_p*Emin_rv                        : coeff= 1

MI after re-optimisation: 1.859332

===== Refinement Iteration 21 =====
Current MI : 1.859332

Emax_rv                            : coeff= 3.856313  MI(-f)= 2.086185  ΔMI=-0.226852  <-- lowest
Emin_rv                            : coeff= 4.528998  MI(-f)= 1.681860  ΔMI= 0.177472
C_p^2                              : coeff= 4.945603  MI(-f)= 1.743951  ΔMI= 0.115382
R_p^2                              : coeff=-5.121598  MI(-f)= 1.710871  ΔMI= 0.148461
log(C_p)                           : coeff= 5.680355  MI(-f)= 1.709969  ΔMI= 0.149364
log(Emax_rv)                       : coeff= 7.648220  MI(-f)= 1.655084  ΔMI= 0.204249
log(C_s)                           : coeff= 5.458805  MI(-f)= 1.729121  ΔMI= 0.130211
Emax_rv*R_s                        : coeff= 10.238151  MI(-f)= 1.152402  ΔMI= 0.706930

Removing: Emax_rv
ΔMI = -0.226852
MI after re-optimisation: 1.852360

===== Refinement Iteration 22 =====
Current MI : 1.852360

Emin_rv                            : coeff= 4.529182  MI(-f)= 1.633830  ΔM

In [26]:
Theta_v_lv_max_cleaned, \
feature_names_v_lv_max_cleaned = build_function_library(
    X_cleaned_train,
    param_names
)

scaler_v_lv_max_cleaned = StandardScaler()

Theta_scaled_v_lv_max_cleaned = (
    scaler_v_lv_max_cleaned.fit_transform(
        Theta_v_lv_max_cleaned
    )
)

sparse_result_v_lv_max_final = sparse_ee_interpretation(
    Theta_scaled_v_lv_max_final,
    y_v_lv_max_final_train,
    feature_names_v_lv_max_final,
    scaler_v_lv_max_final,
    threshold=0.1,
    resume=True
)

save_sparse_result(
    sparse_result_v_lv_max_cleaned,
    filename="../results/sparse_result_v_lv_max_cleaned.pkl"
)


===== Iteration 1 =====
MI              : 2.543355
Active features : 75
C_p                                :  1.728151  KEEP
Za_p                               : -2.248532  KEEP
R_p                                :  9.969474  KEEP
Emax_rv                            : -0.064222  REMOVE
Emin_rv                            : -0.541987  KEEP
C_s                                :  5.312689  KEEP
Za_s                               : -2.505985  KEEP
R_s                                : -2.971618  KEEP
Emax_lv                            :  9.993041  KEEP
Emin_lv                            :  0.246095  KEEP
C_p^2                              : -0.479678  KEEP
Za_p^2                             :  0.265985  KEEP
R_p^2                              :  0.144622  KEEP
Emax_rv^2                          : -1.301949  KEEP
Emin_rv^2                          :  0.181725  KEEP
C_s^2                              : -0.830412  KEEP
Za_s^2                             : -2.201206  KEEP
R_s^2                   


===== Iteration 4 =====
MI              : 1.725376
Active features : 58
C_p                                :  0.645592  KEEP
Za_p                               : -1.950901  KEEP
R_p                                :  5.582530  KEEP
Emin_rv                            : -0.112016  KEEP
C_s                                :  3.866660  KEEP
Za_s                               : -2.097868  KEEP
R_s                                : -2.144100  KEEP
Emax_lv                            :  6.193194  KEEP
C_p^2                              :  0.470528  KEEP
Za_p^2                             :  0.156146  KEEP
R_p^2                              :  1.231767  KEEP
Emax_rv^2                          : -2.210375  KEEP
Emin_rv^2                          : -1.088491  KEEP
C_s^2                              :  0.538016  KEEP
Za_s^2                             : -1.167362  KEEP
R_s^2                              :  0.459413  KEEP
Emax_lv^2                          :  0.354748  KEEP
log(C_p)                  


===== Iteration 7 =====
MI              : 1.746734
Active features : 48
C_p                                :  0.551538  KEEP
R_p                                :  5.904195  KEEP
Emin_rv                            : -1.544652  KEEP
C_s                                :  1.955119  KEEP
Za_s                               : -2.178766  KEEP
R_s                                : -0.473206  KEEP
Emax_lv                            :  6.041467  KEEP
C_p^2                              :  0.688882  KEEP
Za_p^2                             : -1.219288  KEEP
R_p^2                              :  0.339526  KEEP
Emax_rv^2                          : -2.800117  KEEP
Emin_rv^2                          :  0.442839  KEEP
C_s^2                              :  1.674442  KEEP
Za_s^2                             : -0.432188  KEEP
R_s^2                              : -0.829301  KEEP
log(C_p)                           :  1.664736  KEEP
log(R_p)                           :  0.679934  KEEP
log(Emax_rv)              


===== Final Sparse Combination =====
Maximum MI : 1.778421
Terms      : 39
R_p                                : 5.514876
Emin_rv                            : -0.891190
C_s                                : 2.893015
Za_s                               : -2.199492
R_s                                : -1.322501
Emax_lv                            : 6.356778
C_p^2                              : 1.340316
Za_p^2                             : -0.663436
R_p^2                              : 0.885218
Emax_rv^2                          : -2.190002
Emin_rv^2                          : -0.206880
C_s^2                              : 0.694694
Za_s^2                             : -1.000232
log(C_p)                           : 1.298707
log(Emax_rv)                       : -0.615176
log(C_s)                           : 1.590784
log(Emax_lv)                       : 1.422426
C_p*R_p                            : 2.441636
C_p*Emax_rv                        : -0.543829
C_p*Emin_lv                        : 0.35